# Credit Approval: Data Preparation

1. Load and clean the raw data (handle missing values encoded as '?').
2. Encode categorical features using one-hot encoding.
3. Create a single, hold-out test set.
4. From the main training set, generate multiple training datasets with varying **imbalance ratios (IR)** by undersampling the minority class.
5. For each IR, create **multiple repetitions** with different random samples of the minority class.
6. For each IR and repetition, create a size-matched **control dataset** with the original class ratio.
7. Preprocess and save all generated datasets.

## Dataset Information

- **Source:** UCI Machine Learning Repository (1987)
- **Instances:** 690 credit card applications
- **Features:** 15 (6 numerical, 9 categorical)
- **Target:** Credit approval decision (+ = approved, - = denied)
- **Natural Imbalance Ratio:** ~1.25:1 (Denied is slight majority)
- **Missing Values:** Present in 7 features (handled via imputation)

# 1. Dataset Configuration

In [1]:
DATASET_NAME = "credit"

print(f"Dataset: {DATASET_NAME}")

Dataset: credit


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils import resample
from pathlib import Path
import sys

sys.path.append(str(Path("../../").resolve()))
from config.config import get_config

config = get_config()

RAW_PATH = Path("../../data/raw/credit.csv")
PROCESSED_PATH = Path(f"../../data/processed/{DATASET_NAME}/")

COLUMN_NAMES = ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'target']

NUMERICAL_FEATURES = ['A2', 'A3', 'A8', 'A11', 'A14', 'A15']
CATEGORICAL_FEATURES = ['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13']

TARGET_FEATURE = "target"
CLASS_DENIED = 0     # Mapped from '-'
CLASS_APPROVED = 1   # Mapped from '+'

RANDOM_STATE = config.experiment.random_state
IMBALANCE_RATIOS = config.experiment.imbalance_ratios
N_REPETITIONS = config.experiment.n_repetitions

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Configuration:")
print(f"  - Dataset: {DATASET_NAME}")
print(f"  - Raw data path: {RAW_PATH}")
print(f"  - Processed data path: {PROCESSED_PATH}")
print(f"  - Target feature: {TARGET_FEATURE}")
print(f"  - Numerical features: {len(NUMERICAL_FEATURES)}")
print(f"  - Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"  - Imbalance ratios: {IMBALANCE_RATIOS}")
print(f"  - Repetitions per IR: {N_REPETITIONS}")
print(f"  - Random state: {RANDOM_STATE}")

Configuration:
  - Dataset: credit
  - Raw data path: ../../data/raw/credit.csv
  - Processed data path: ../../data/processed/credit
  - Target feature: target
  - Numerical features: 6
  - Categorical features: 9
  - Imbalance ratios: [1, 3, 5, 7, 10, 20, 50, 100]
  - Repetitions per IR: 10
  - Random state: 42


In [3]:
df_raw = pd.read_csv(RAW_PATH, names=COLUMN_NAMES, na_values='?')

print(f"Raw dataset loaded. Shape: {df_raw.shape}")
print(f"\nFirst 5 rows:")
df_raw.head()

Raw dataset loaded. Shape: (690, 16)

First 5 rows:


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,target
0,b,30.83,0.000,u,g,w,v,1.25,t,t,1,f,g,202.0,0,+
1,a,58.67,4.460,u,g,q,h,3.04,t,t,6,f,g,43.0,560,+
2,a,24.50,0.500,u,g,q,h,1.50,t,f,0,f,g,280.0,824,+
3,b,27.83,1.540,u,g,w,v,3.75,t,t,5,t,g,100.0,3,+
4,b,20.17,5.625,u,g,w,v,1.71,t,f,0,f,s,120.0,0,+


In [4]:
print(f"\nData types:")
print(df_raw.dtypes)


Data types:
A1         object
A2        float64
A3        float64
A4         object
A5         object
A6         object
A7         object
A8        float64
A9         object
A10        object
A11         int64
A12        object
A13        object
A14       float64
A15         int64
target     object
dtype: object


# 2. Data Cleaning and Target Encoding

The Credit Approval dataset has:
1. Missing values in 7 features (encoded as '?' in raw data, now NaN)
2. Target encoded as '+' (approved) and '-' (denied)

We need to:
1. Encode the target variable to binary (+ -> 1, - -> 0)
2. Handle missing values via imputation

In [5]:
print("Missing values per column:")
missing = df_raw.isnull().sum()
print(f"Total missing values: {missing.sum()}")
print("\nColumns with missing values:")
print(missing[missing > 0])

Missing values per column:
Total missing values: 67

Columns with missing values:
A1     12
A2     12
A4      6
A5      6
A6      9
A7      9
A14    13
dtype: int64


In [6]:
# Check unique values in the target
print(f"\nOriginal target unique values: {sorted(df_raw['target'].dropna().unique())}")
print(f"\nOriginal target distribution:")
print(df_raw['target'].value_counts().sort_index())


Original target unique values: ['+', '-']

Original target distribution:
target
+    307
-    383
Name: count, dtype: int64


In [7]:
# Encode target: + (Approved) -> 1, - (Denied) -> 0
df_cleaned = df_raw.copy()

df_cleaned[TARGET_FEATURE] = df_cleaned[TARGET_FEATURE].map({'+': 1, '-': 0})

print(f"Target variable re-encoded:")
print(f"  + (Approved) -> 1")
print(f"  - (Denied)   -> 0")

print(f"\nNew target distribution:")
print(df_cleaned[TARGET_FEATURE].value_counts().sort_index())

Target variable re-encoded:
  + (Approved) -> 1
  - (Denied)   -> 0

New target distribution:
target
0    383
1    307
Name: count, dtype: int64


# 3. Missing Value Imputation

Strategy:
- **Numerical features:** Impute with median (robust to outliers)
- **Categorical features:** Impute with mode (most frequent value)

In [8]:
features_with_missing = missing[missing > 0].index.tolist()
if TARGET_FEATURE in features_with_missing:
    features_with_missing.remove(TARGET_FEATURE)

print(f"Features with missing values: {features_with_missing}")

# Separate by type
num_missing = [f for f in features_with_missing if f in NUMERICAL_FEATURES]
cat_missing = [f for f in features_with_missing if f in CATEGORICAL_FEATURES]

print(f"\nNumerical features with missing: {num_missing}")
print(f"Categorical features with missing: {cat_missing}")

Features with missing values: ['A1', 'A2', 'A4', 'A5', 'A6', 'A7', 'A14']

Numerical features with missing: ['A2', 'A14']
Categorical features with missing: ['A1', 'A4', 'A5', 'A6', 'A7']


In [9]:
# Impute numerical features with median
if num_missing:
    num_imputer = SimpleImputer(strategy='median')
    df_cleaned[num_missing] = num_imputer.fit_transform(df_cleaned[num_missing])
    print(f"Imputed numerical features with median: {num_missing}")
    for col in num_missing:
        print(f"  {col}: median = {num_imputer.statistics_[num_missing.index(col)]:.2f}")

# Impute categorical features with mode
if cat_missing:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df_cleaned[cat_missing] = cat_imputer.fit_transform(df_cleaned[cat_missing])
    print(f"\nImputed categorical features with mode: {cat_missing}")
    for col in cat_missing:
        print(f"  {col}: mode = '{cat_imputer.statistics_[cat_missing.index(col)]}'")

Imputed numerical features with median: ['A2', 'A14']
  A2: median = 28.46
  A14: median = 160.00

Imputed categorical features with mode: ['A1', 'A4', 'A5', 'A6', 'A7']
  A1: mode = 'b'
  A4: mode = 'u'
  A5: mode = 'g'
  A6: mode = 'c'
  A7: mode = 'v'


In [10]:
remaining_missing = df_cleaned.isnull().sum().sum()
print(f"\nRemaining missing values: {remaining_missing}")
assert remaining_missing == 0, "Missing values still present!"
print("All missing values have been imputed.")


Remaining missing values: 0
All missing values have been imputed.


# 4. Categorical Feature Encoding

We use one-hot encoding for categorical features to create binary indicator columns. This is preferred for:
- Tree-based models (Random Forest)
- Distance-based algorithms
- Neural networks

In [11]:
print("Categorical feature cardinality:")
for col in CATEGORICAL_FEATURES:
    unique_vals = df_cleaned[col].unique()
    print(f"  {col}: {len(unique_vals)} unique values -> {list(unique_vals)}")

Categorical feature cardinality:
  A1: 2 unique values -> ['b', 'a']
  A4: 3 unique values -> ['u', 'y', 'l']
  A5: 3 unique values -> ['g', 'p', 'gg']
  A6: 14 unique values -> ['w', 'q', 'm', 'r', 'cc', 'k', 'c', 'd', 'x', 'i', 'e', 'aa', 'ff', 'j']
  A7: 9 unique values -> ['v', 'h', 'bb', 'ff', 'j', 'z', 'o', 'dd', 'n']
  A9: 2 unique values -> ['t', 'f']
  A10: 2 unique values -> ['t', 'f']
  A12: 2 unique values -> ['f', 't']
  A13: 3 unique values -> ['g', 's', 'p']


In [12]:
# One-hot encode categorical features
df_encoded = pd.get_dummies(df_cleaned, columns=CATEGORICAL_FEATURES, drop_first=False, dtype=float)

print(f"\nShape before encoding: {df_cleaned.shape}")
print(f"Shape after encoding: {df_encoded.shape}")
print(f"\nNew columns ({len(df_encoded.columns) - len(NUMERICAL_FEATURES) - 1} encoded features):")

encoded_cols = [col for col in df_encoded.columns if col not in NUMERICAL_FEATURES and col != TARGET_FEATURE]
print(f"  {encoded_cols[:10]}... (and {len(encoded_cols) - 10} more)" if len(encoded_cols) > 10 else f"  {encoded_cols}")


Shape before encoding: (690, 16)
Shape after encoding: (690, 47)

New columns (40 encoded features):
  ['A1_a', 'A1_b', 'A4_l', 'A4_u', 'A4_y', 'A5_g', 'A5_gg', 'A5_p', 'A6_aa', 'A6_c']... (and 30 more)


In [13]:
ALL_FEATURES = [col for col in df_encoded.columns if col != TARGET_FEATURE]

print(f"Total features after encoding: {len(ALL_FEATURES)}")
print(f"  - Original numerical: {len(NUMERICAL_FEATURES)}")
print(f"  - Encoded categorical: {len(ALL_FEATURES) - len(NUMERICAL_FEATURES)}")

df_encoded.head()

Total features after encoding: 46
  - Original numerical: 6
  - Encoded categorical: 40


,A2,A3,A8,A11,A14,A15,target,A1_a,A1_b,A4_l,...,A7_z,A9_f,A9_t,A10_f,A10_t,A12_f,A12_t,A13_g,A13_p,A13_s
0,30.83,0.000,1.25,1,202.0,0,1,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
1,58.67,4.460,3.04,6,43.0,560,1,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
2,24.50,0.500,1.50,0,280.0,824,1,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3,27.83,1.540,3.75,5,100.0,3,1,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
4,20.17,5.625,1.71,0,120.0,0,1,0.0,1.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


# 5. Confirming Majority and Minority Classes

For our imbalance experiments:
- **Class 0 (Denied):** Majority class
- **Class 1 (Approved):** Minority class

In [14]:
print("Target variable distribution:")
print(df_encoded[TARGET_FEATURE].value_counts().sort_index())

n_denied = df_encoded[TARGET_FEATURE].value_counts()[CLASS_DENIED]
n_approved = df_encoded[TARGET_FEATURE].value_counts()[CLASS_APPROVED]

print(f"\nClass balance:")
print(f"  - Denied (0):   {n_denied:,} ({n_denied/len(df_encoded)*100:.2f}%)")
print(f"  - Approved (1): {n_approved:,} ({n_approved/len(df_encoded)*100:.2f}%)")

print(f"\nMajority class: Denied (0)")
print(f"Minority class: Approved (1)")
CLASS_MAJORITY = CLASS_DENIED
CLASS_MINORITY = CLASS_APPROVED

natural_ir = max(n_denied, n_approved) / min(n_denied, n_approved)
print(f"\nNatural imbalance ratio: {natural_ir:.2f}:1")

Target variable distribution:
target
0    383
1    307
Name: count, dtype: int64

Class balance:
  - Denied (0):   383 (55.51%)
  - Approved (1): 307 (44.49%)

Majority class: Denied (0)
Minority class: Approved (1)

Natural imbalance ratio: 1.25:1


# 6. Create a Hold-Out Test Set

We perform a one-time stratified split to create a final test set. All experimental datasets will be generated from the `train_full_df`.

In [15]:
X = df_encoded.drop(TARGET_FEATURE, axis=1)
y = df_encoded[[TARGET_FEATURE]]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=config.experiment.test_size,
    random_state=RANDOM_STATE,
    stratify=y
)

train_full_df = pd.concat([X_train_full, y_train_full], axis=1)

print(f"Full training set shape: {train_full_df.shape}")
print(f"Hold-out test set shape: {X_test.shape}")
print(f"\nTraining set class distribution:")
print(train_full_df[TARGET_FEATURE].value_counts().sort_index())

n_train_majority = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MAJORITY].shape[0]
n_train_minority = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MINORITY].shape[0]
print(f"\nTraining set: {n_train_majority:,} majority, {n_train_minority:,} minority")

Full training set shape: (552, 47)
Hold-out test set shape: (138, 46)

Training set class distribution:
target
0    306
1    246
Name: count, dtype: int64

Training set: 306 majority, 246 minority


# 7. Generate Imbalanced and Control Datasets with Multiple Repetitions

For each IR, we create multiple repetitions by sampling different subsets of the minority class. This allows us to test whether methods work reliably across different minority class samples.

**Methodology:**
1. Start with the **full majority class** (Denied applications).
2. **Undersample the minority class** (Approved applications) to achieve the desired Imbalance Ratio (IR).
3. **Repeat this sampling N_REPETITIONS times** with different random seeds.
4. Create a size-matched **control dataset** for each IR and repetition.

In [16]:
minority_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MINORITY]
majority_df = train_full_df[train_full_df[TARGET_FEATURE] == CLASS_MAJORITY]
n_minority_available = len(minority_df)
n_majority_available = len(majority_df)

print(f"\nFull training set composition:")
print(f"  - Majority (Denied): {n_majority_available:,}")
print(f"  - Minority (Approved): {n_minority_available:,}")
print(f"\nGenerating datasets with {N_REPETITIONS} repetitions per imbalance ratio...")

generated_datasets = {}
skipped_ratios = []

for ir in IMBALANCE_RATIOS:
    print(f"\nProcessing Imbalance Ratio (IR) = {ir}:1")
    
    for rep_id in range(1, N_REPETITIONS + 1):
        rep_seed = RANDOM_STATE + (ir * 1000) + rep_id
        
        if ir == 1:
            # For IR=1:1, undersample majority to match minority
            majority_undersampled = resample(
                majority_df,
                replace=False,
                n_samples=n_minority_available, 
                random_state=rep_seed 
            )
            imbalanced_df = pd.concat([majority_undersampled, minority_df])
            
        else:
            # For IR > 1, keep full majority and undersample minority
            majority_full_set = majority_df
            
            n_minority_imbalanced = int(n_majority_available / ir)

            if n_minority_imbalanced > n_minority_available:
                if rep_id == 1:  
                    print(f"  SKIPPING IR={ir}: Requires {n_minority_imbalanced} minority samples but only {n_minority_available} available.")
                    skipped_ratios.append(ir)
                continue
            if n_minority_imbalanced < 1:
                if rep_id == 1:
                    print(f"  SKIPPING IR={ir}: Would result in zero minority samples.")
                    skipped_ratios.append(ir)
                continue

            minority_undersampled = resample(
                minority_df,
                replace=False,
                n_samples=n_minority_imbalanced,
                random_state=rep_seed 
            )

            imbalanced_df = pd.concat([majority_full_set, minority_undersampled])

        total_size = len(imbalanced_df)
        
        dataset_key = f'imbalanced_ir_{ir}_rep{rep_id}'
        generated_datasets[dataset_key] = imbalanced_df
        
        n_maj = len(imbalanced_df[imbalanced_df[TARGET_FEATURE] == CLASS_MAJORITY])
        n_min = len(imbalanced_df[imbalanced_df[TARGET_FEATURE] == CLASS_MINORITY])
        
        if rep_id == 1: 
            print(f"  Rep {rep_id}: {total_size:,} samples ({n_maj:,} majority, {n_min:,} minority)")

        # Create control dataset with original class ratio
        if total_size >= len(train_full_df):
            control_df = train_full_df.copy()
        else:
            control_df, _ = train_test_split(
                train_full_df,
                train_size=total_size,
                random_state=rep_seed,  
                stratify=train_full_df[TARGET_FEATURE]
            )
        
        control_key = f'control_ir_{ir}_rep{rep_id}'
        generated_datasets[control_key] = control_df

print(f"Total datasets created: {len(generated_datasets)}")
print(f"  - Imbalanced: {len([k for k in generated_datasets.keys() if 'imbalanced' in k])}")
print(f"  - Control: {len([k for k in generated_datasets.keys() if 'control' in k])}")

if skipped_ratios:
    print(f"\nSkipped imbalance ratios (insufficient minority samples): {list(set(skipped_ratios))}")


Full training set composition:
  - Majority (Denied): 306
  - Minority (Approved): 246

Generating datasets with 10 repetitions per imbalance ratio...

Processing Imbalance Ratio (IR) = 1:1
  Rep 1: 492 samples (246 majority, 246 minority)

Processing Imbalance Ratio (IR) = 3:1
  Rep 1: 408 samples (306 majority, 102 minority)

Processing Imbalance Ratio (IR) = 5:1
  Rep 1: 367 samples (306 majority, 61 minority)

Processing Imbalance Ratio (IR) = 7:1
  Rep 1: 349 samples (306 majority, 43 minority)

Processing Imbalance Ratio (IR) = 10:1
  Rep 1: 336 samples (306 majority, 30 minority)

Processing Imbalance Ratio (IR) = 20:1
  Rep 1: 321 samples (306 majority, 15 minority)

Processing Imbalance Ratio (IR) = 50:1
  Rep 1: 312 samples (306 majority, 6 minority)

Processing Imbalance Ratio (IR) = 100:1
  Rep 1: 309 samples (306 majority, 3 minority)
Total datasets created: 160
  - Imbalanced: 80
  - Control: 80


In [17]:


summary_data = []
for name in sorted(generated_datasets.keys()):
    df_temp = generated_datasets[name]
    n_total = len(df_temp)
    n_maj = len(df_temp[df_temp[TARGET_FEATURE] == CLASS_MAJORITY])
    n_min = len(df_temp[df_temp[TARGET_FEATURE] == CLASS_MINORITY])
    actual_ir = n_maj / n_min if n_min > 0 else float('inf')
    
    summary_data.append({
        'Dataset': name,
        'Total': n_total,
        'Majority': n_maj,
        'Minority': n_min,
        'Actual IR': f"{actual_ir:.2f}:1"
    })

summary_df = pd.DataFrame(summary_data)

# Show first 10 datasets as sample
print("\nFirst 10 datasets:")
display(summary_df.head(10))

print(f"\n... and {len(summary_df) - 10} more datasets")


First 10 datasets:


,Dataset,Total,Majority,Minority,Actual IR
0,control_ir_100_rep1,309,171,138,1.24:1
1,control_ir_100_rep10,309,171,138,1.24:1
2,control_ir_100_rep2,309,171,138,1.24:1
3,control_ir_100_rep3,309,171,138,1.24:1
4,control_ir_100_rep4,309,171,138,1.24:1
5,control_ir_100_rep5,309,171,138,1.24:1
6,control_ir_100_rep6,309,171,138,1.24:1
7,control_ir_100_rep7,309,171,138,1.24:1
8,control_ir_100_rep8,309,171,138,1.24:1
9,control_ir_100_rep9,309,171,138,1.24:1



... and 150 more datasets


# 8. Preprocessing and Saving All Datasets

We apply StandardScaler to all features (both original numerical and one-hot encoded categorical).

We fit the scaler **once** on the full training data. Then, we transform all generated training sets and the hold-out test set using this single, consistent scaler.

In [18]:
FEATURES_TO_SCALE = ALL_FEATURES

print(f"Features to scale ({len(FEATURES_TO_SCALE)}):")
print(f"  Numerical ({len(NUMERICAL_FEATURES)}): {NUMERICAL_FEATURES}")
print(f"  Encoded categorical ({len(FEATURES_TO_SCALE) - len(NUMERICAL_FEATURES)}): [one-hot encoded]")

Features to scale (46):
  Numerical (6): ['A2', 'A3', 'A8', 'A11', 'A14', 'A15']
  Encoded categorical (40): [one-hot encoded]


In [19]:
scaler = StandardScaler()
scaler.fit(X_train_full[FEATURES_TO_SCALE])

print("Scaler fitted on full training data.")
print(f"\nScaler statistics (first 6 numerical features):")
for i, feat in enumerate(NUMERICAL_FEATURES):
    feat_idx = FEATURES_TO_SCALE.index(feat)
    print(f"  {feat}: mean={scaler.mean_[feat_idx]:.4f}, std={scaler.scale_[feat_idx]:.4f}")

Scaler fitted on full training data.

Scaler statistics (first 6 numerical features):
  A2: mean=31.4799, std=11.6917
  A3: mean=4.7880, std=5.0934
  A8: mean=2.1692, std=3.3433
  A11: mean=2.4946, std=5.0652
  A14: mean=180.3043, std=159.5067
  A15: mean=1040.5326, std=5364.6543


In [20]:
print("Scaling and saving datasets...\n")

saved_count = 0
for name, df in generated_datasets.items():
    X_temp = df.drop(columns=[TARGET_FEATURE])
    y_temp = df[[TARGET_FEATURE]]

    X_processed = scaler.transform(X_temp[FEATURES_TO_SCALE])
    X_processed_df = pd.DataFrame(X_processed, columns=FEATURES_TO_SCALE)
    
    final_df = X_processed_df.reset_index(drop=True)
    final_df[TARGET_FEATURE] = y_temp.reset_index(drop=True)
    
    save_path = PROCESSED_PATH / f"train_{name}.csv"
    final_df.to_csv(save_path, index=False)
    saved_count += 1
    
    # Print progress every 20 files
    if saved_count % 20 == 0:
        print(f"  Saved {saved_count} files...")

print(f"\nSaved all {saved_count} training files.")

Scaling and saving datasets...

  Saved 20 files...
  Saved 40 files...
  Saved 60 files...
  Saved 80 files...
  Saved 100 files...
  Saved 120 files...
  Saved 140 files...
  Saved 160 files...

Saved all 160 training files.


In [21]:
X_test_processed = scaler.transform(X_test[FEATURES_TO_SCALE])
X_test_processed_df = pd.DataFrame(X_test_processed, columns=FEATURES_TO_SCALE)

test_df = X_test_processed_df.reset_index(drop=True)
test_df[TARGET_FEATURE] = y_test.reset_index(drop=True)
test_df.to_csv(PROCESSED_PATH / "test.csv", index=False)

print(f"Saved test set: test.csv ({len(test_df):,} samples)")
print(f"\nTest set class distribution:")
print(test_df[TARGET_FEATURE].value_counts().sort_index())

Saved test set: test.csv (138 samples)

Test set class distribution:
target
0    77
1    61
Name: count, dtype: int64


# 9. Save Metadata

In [22]:
import json
from datetime import datetime

created_irs = sorted(list(set([int(k.split('_')[2]) for k in generated_datasets.keys() if 'imbalanced' in k])))

metadata = {
    "dataset_name": DATASET_NAME,
    "openml_id": 29,
    "source": "UCI Machine Learning Repository",
    "target_feature": TARGET_FEATURE,
    "original_target_encoding": {"+": "Approved (1)", "-": "Denied (0)"},
    "processing_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "n_train_files": len(generated_datasets),
    "imbalance_ratios_requested": IMBALANCE_RATIOS,
    "imbalance_ratios_created": created_irs,
    "n_repetitions": N_REPETITIONS,
    "random_state": RANDOM_STATE,
    "test_size": len(test_df),
    "train_full_size": len(train_full_df),
    "n_original_features": 15,
    "n_features_after_encoding": len(FEATURES_TO_SCALE),
    "numerical_features": NUMERICAL_FEATURES,
    "categorical_features_original": CATEGORICAL_FEATURES,
    "all_features": FEATURES_TO_SCALE,
    "class_mapping": {
        "0": "Denied (majority)",
        "1": "Approved (minority)"
    },
    "missing_value_handling": {
        "strategy": "imputation",
        "numerical": "median",
        "categorical": "mode"
    },
    "natural_imbalance_ratio": round(natural_ir, 2),
    "notes": "Credit card approval dataset - mixed numerical and categorical features, all attributes anonymized"
}

metadata_path = PROCESSED_PATH / "metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")
print("\nMetadata contents:")
for key, value in metadata.items():
    if key not in ['features', 'all_features', 'numerical_features', 'categorical_features_original']:  # Don't print all features
        print(f"  {key}: {value}")

Metadata saved to: ../../data/processed/credit/metadata.json

Metadata contents:
  dataset_name: credit
  openml_id: 29
  source: UCI Machine Learning Repository
  target_feature: target
  original_target_encoding: {'+': 'Approved (1)', '-': 'Denied (0)'}
  processing_timestamp: 2026-06-23 10:39:12
  n_train_files: 160
  imbalance_ratios_requested: [1, 3, 5, 7, 10, 20, 50, 100]
  imbalance_ratios_created: [1, 3, 5, 7, 10, 20, 50, 100]
  n_repetitions: 10
  random_state: 42
  test_size: 138
  train_full_size: 552
  n_original_features: 15
  n_features_after_encoding: 46
  class_mapping: {'0': 'Denied (majority)', '1': 'Approved (minority)'}
  missing_value_handling: {'strategy': 'imputation', 'numerical': 'median', 'categorical': 'mode'}
  natural_imbalance_ratio: 1.25
  notes: Credit card approval dataset - mixed numerical and categorical features, all attributes anonymized


In [23]:
saved_files = list(PROCESSED_PATH.glob("*.csv"))
print(f"  - Total CSV files saved: {len(saved_files)}")
print(f"  - Expected: {len(generated_datasets) + 1} (training + test)")

total_size_mb = sum(f.stat().st_size for f in saved_files) / (1024 * 1024)
print(f"  - Total size: {total_size_mb:.2f} MB")

  - Total CSV files saved: 161
  - Expected: 161 (training + test)
  - Total size: 51.25 MB


In [24]:
# Load and verify a sample file
sample_file = PROCESSED_PATH / "train_imbalanced_ir_1_rep1.csv"
if sample_file.exists():
    sample_df = pd.read_csv(sample_file)
    print(f"\nSample verification (train_imbalanced_ir_1_rep1.csv):")
    print(f"  Shape: {sample_df.shape}")
    print(f"  Columns: {list(sample_df.columns)[:5]}... (and {len(sample_df.columns)-5} more)")
    print(f"  Target distribution:")
    print(sample_df[TARGET_FEATURE].value_counts().sort_index())
    print(f"\n  Feature statistics (first 3 numerical):")
    print(sample_df[NUMERICAL_FEATURES[:3]].describe().round(3))


Sample verification (train_imbalanced_ir_1_rep1.csv):
  Shape: (492, 47)
  Columns: ['A2', 'A3', 'A8', 'A11', 'A14']... (and 42 more)
  Target distribution:
target
0    246
1    246
Name: count, dtype: int64

  Feature statistics (first 3 numerical):
            A2       A3       A8
count  492.000  492.000  492.000
mean     0.008    0.020    0.038
std      1.015    1.006    1.038
min     -1.395   -0.940   -0.649
25%     -0.754   -0.744   -0.599
50%     -0.258   -0.404   -0.350
75%      0.517    0.551    0.143
max      4.171    4.557    7.876


In [25]:

print(f"\nDataset: {DATASET_NAME}")
print(f"\nOriginal data:")
print(f"  - Total instances: {len(df_cleaned):,}")
print(f"  - Original features: 15 (6 numerical, 9 categorical)")
print(f"  - Features after encoding: {len(FEATURES_TO_SCALE)}")
print(f"  - Natural IR: {natural_ir:.2f}:1")

print(f"\nData cleaning:")
print(f"  - Missing values imputed: {missing.sum()} cells")
print(f"  - Categorical features one-hot encoded")

print(f"\nGenerated datasets:")
print(f"  - Training files: {len(generated_datasets)}")
print(f"  - Test file: 1")
print(f"  - Imbalance ratios: {created_irs}")
print(f"  - Repetitions per IR: {N_REPETITIONS}")

print(f"\nOutput location: {PROCESSED_PATH}")


Dataset: credit

Original data:
  - Total instances: 690
  - Original features: 15 (6 numerical, 9 categorical)
  - Features after encoding: 46
  - Natural IR: 1.25:1

Data cleaning:
  - Missing values imputed: 67 cells
  - Categorical features one-hot encoded

Generated datasets:
  - Training files: 160
  - Test file: 1
  - Imbalance ratios: [1, 3, 5, 7, 10, 20, 50, 100]
  - Repetitions per IR: 10

Output location: ../../data/processed/credit
